In [1]:
%pip install -Uq chromadb llama-index llama-cloud-services

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.0/362.0 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/20

In [2]:
import os
import getpass

os.environ["LLAMA_CLOUD_API_KEY"] = getpass.getpass("Llama Cloud API Key:")

Llama Cloud API Key:··········


In [5]:
import chromadb

PERSISTENT_DIR = "./chroma_db"
COLLECTION_NAME="rag_mcp"

def init_chroma():
    client = chromadb.PersistentClient(path=PERSISTENT_DIR)

    collection = client.get_or_create_collection(name=COLLECTION_NAME)
    return collection

init_chroma()

def get_chroma_client():
    return chromadb.PersistentClient(path=PERSISTENT_DIR)


In [6]:
# get some files to play with

!mkdir -p 'data'
!wget 'https://arxiv.org/pdf/2511.13648' -O 'data/paper1.pdf'
!wget 'https://arxiv.org/pdf/2511.10647' -O 'data/paper2.pdf'
!wget 'https://arxiv.org/pdf/2511.13720' -O 'data/paper3.pdf'

--2026-06-24 08:24:22--  https://arxiv.org/pdf/2511.13648
Resolving arxiv.org (arxiv.org)... 151.101.3.42, 151.101.67.42, 151.101.195.42, ...
Connecting to arxiv.org (arxiv.org)|151.101.3.42|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4994615 (4.8M) [application/pdf]
Saving to: ‘data/paper1.pdf’

data/paper1.pdf     100%[===================>]   4.76M  7.39MB/s    in 0.6s    

2026-06-24 08:24:23 (7.39 MB/s) - ‘data/paper1.pdf’ saved [4994615/4994615]

--2026-06-24 08:24:23--  https://arxiv.org/pdf/2511.10647
Resolving arxiv.org (arxiv.org)... 151.101.3.42, 151.101.195.42, 151.101.67.42, ...
Connecting to arxiv.org (arxiv.org)|151.101.3.42|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 20403091 (19M) [application/pdf]
Saving to: ‘data/paper2.pdf’

data/paper2.pdf     100%[===================>]  19.46M  9.57MB/s    in 2.0s    

2026-06-24 08:24:25 (9.57 MB/s) - ‘data/paper2.pdf’ saved [20403091/20403091]

--2026-06-24 08:24:25--  h

In [7]:
from llama_index.core import SimpleDirectoryReader
from llama_cloud_services import LlamaParse

DATA_DIR = "./data"

LLAMA_CLOUD_API_KEY = os.environ["LLAMA_CLOUD_API_KEY"]

def ingest_data_dir():
    chroma_client = get_chroma_client()
    chroma_client.delete_collection(name=COLLECTION_NAME)
    collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)

    parser = LlamaParse(api_key=LLAMA_CLOUD_API_KEY, result_type="text")

    file_extractor = {".pdf": parser}

    documents = SimpleDirectoryReader(DATA_DIR, file_extractor=file_extractor).load_data()

    for doc in documents:
        collection.add(
            documents=[doc.text],
            metadatas=[doc.metadata],
            ids=[doc.doc_id]
        )

    final_count = collection.count()
    print(f"Ingested {final_count} documents")


ingest_data_dir()


/tmp/ipykernel_6772/1626816799.py:2: DeprecationWarning: This package (llama-cloud-services) is deprecated and will be maintained until May 1, 2026. Please migrate to the new package: pip install llama-cloud>=1.0 (https://github.com/run-llama/llama-cloud-py). The new package provides the same functionality with improved performance and support.
  from llama_cloud_services import LlamaParse


Started parsing the file under job_id f7b464d7-bb23-4cab-a92c-773a7b08c8f6


Error while parsing the file '<bytes/buffer>': Event loop is closed
Started parsing the file under job_id 1be7eccc-d5cb-4a99-b85c-ecafca2dbd68


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:03<00:00, 26.2MiB/s]


Ingested 28 documents


In [8]:

def query_documents(query: str, n_results: int = 2) -> str:
    chroma_client = get_chroma_client()
    collection = chroma_client.get_collection(name=COLLECTION_NAME)

    results = collection.query(
        query_texts=["PhysX-Anything?"],
        n_results=2,
        include=["metadatas", "documents", "distances"]
    )

    if len(results["documents"]) == 0 or not results["documents"][0]:
        return "No documents found"

    # Format results
    formatted_results = []
    documents = results["documents"][0]
    metadatas = results["metadatas"][0] if results["metadatas"] else [{}] * len(documents)
    distances = results["distances"][0] if results["distances"] else [0] * len(documents)

    for i, (doc, metadata, distance) in enumerate(zip(documents, metadatas, distances)):
        result_text = f"\n--- Result {i+1} ---\n"
        result_text += f"Content: {doc}\n"
        result_text += f"Source: {metadata.get('file_name', 'Unknown')}\n"
        result_text += f"Similarity Score: {1 - distance:.3f}\n"
        formatted_results.append(result_text)

    response = f"Found {len(documents)} relevant documents for query: '{query}'\n"
    response += "\n".join(formatted_results)

    return response

results = query_documents("What is PhysX-Anything?")
print(results)

Found 2 relevant documents for query: 'What is PhysX-Anything?'

--- Result 1 ---
Content:     PhysX-Anything: Simulation-Ready Physical 3D Assets from Single Image


    Ziang Cao¹, Fangzhou Hong¹, Zhaoxi Chen¹, Liang Pan², Ziwei Liu¹*
    1 S-Lab, Nanyang Technological University  2 Shanghai AI Lab

    https://physx-anything.github.io



  arXiv:2511.13648v1 [cs.CV] 17 Nov 2025
    Revolute





 Revolute
& Prismatic

    Revolute



    Revolute






                                                                                                                Revolute
    Input             Output (PhysX-Anything)    Input             Output (PhysX-Anything)    Input             Output (PhysX-Anything)    Input    Output (PhysX-Anything)
             #1       #19  #6 1    #76                    #1       #24  #6 7    #80                    #1       #17  #63  #79                       #1  #21  #67  #81
             Free                                         Free                   